# PIP Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Who is your pip.** `python -m pip` guarantees the packages land in the interpreter you actually invoked.

In [ ]:
# Terminal commands - run in PowerShell / cmd / Git Bash, NOT inside Python:
#
# pip --version
# pip 25.2 from H:\...\site-packages\pip (python 3.x)
#
# python -m pip --version        <- preferred: pins the EXACT interpreter
# pip 25.2 from H:\...\site-packages\pip (python 3.x)

# Cross-check pip's own registration from inside Python:
from importlib.metadata import version
print("pip", version("pip"), "is installed for THIS interpreter")

**2. Fetch a package.** `pip install` fetches from PyPI, resolves dependencies and unpacks into site-packages.

In [ ]:
# python -m pip install requests                  # newest stable
# python -m pip install requests==2.32.3          # exact pin - reproducible
# python -m pip install numpy pandas matplotlib   # several at once

# Prove what landed (adjust the list to whatever you installed):
from importlib.metadata import version

for pkg in ["requests", "numpy"]:
    try:
        print(pkg, version(pkg))
    except Exception:
        print(pkg, "not installed here")

**3. Upgrade, then remove.** `-U` means upgrade; `-y` auto-confirms the uninstall. There is NO update-all — careful projects upgrade deliberately, one package at a time.

In [ ]:
# python -m pip install -U requests      # -U = --upgrade, newest allowed
# python -m pip uninstall requests       # asks for confirmation
# python -m pip uninstall -y requests    # -y skips the prompt (script-friendly)
#
# No "update-all" exists - upgrade deliberately, then re-run your tests.

## Part 2 — Practice

**4. Take inventory.** `pip list` / `pip show` read the dist-info registry; `site.getsitepackages()` reveals where that registry lives.

In [ ]:
# python -m pip list                # every installed package + version
# python -m pip list --outdated     # which ones have newer releases
# python -m pip show numpy          # version, author, homepage, dependencies

import site
import sys

print("packages land in:", site.getsitepackages()[0])
print("this interpreter :", sys.executable)

**5. Inventory from inside Python.** `importlib.metadata` queries the same registry pip maintains — no terminal needed.

In [ ]:
from importlib.metadata import version, metadata

for pkg in ["numpy", "pandas", "matplotlib"]:
    try:
        print(f"{pkg:<12} {version(pkg)}")
    except Exception:
        print(f"{pkg:<12} not installed in this environment")

info = metadata("pip")
print("even pip itself:", info["Name"], info["Version"])

**6. The environment recipe.** `pip freeze > requirements.txt` snapshots; `pip install -r requirements.txt` restores. Commit the recipe, never the packages.

In [ ]:
from pathlib import Path

Path("requirements.txt").write_text(
    "numpy==2.3.1\n"      # hard pin - maximum reproducibility
    "pandas>=2.2,<3\n"    # floor AND ceiling - the common sweet spot
    "requests~=2.32\n"    # >=2.32, <3.0 - compatible within the major
    "matplotlib\n",       # any version - fine for toys, risky for teams
    encoding="utf-8",
)
print(Path("requirements.txt").read_text(encoding="utf-8"))

# Generate from your current environment:
#   python -m pip freeze > requirements.txt
# Rebuild the identical environment anywhere:
#   python -m pip install -r requirements.txt

## Part 3 — Challenge

**7. Guided tour: rich.** Every step uses `python -m pip` so installs stay attached to the intended interpreter.

In [ ]:
# The full round trip, in order:
#
# python -m pip --version                    # 1. who am I using?
# python -m pip install rich                 # 2. fetch a pretty-terminal library
# python -m pip show rich                    # 3. inspect it
# python -m rich                             # 4. its built-in demo
# python -m pip freeze > requirements.txt    # 5. snapshot the environment
# type requirements.txt                      #    (Windows) peek inside
# python -m pip uninstall -y rich            # 6. clean up

from importlib.metadata import version

try:
    print("rich", version("rich"), "is installed right now")
except Exception:
    print("rich is NOT installed right now")

**8. Specifier whisperer.** `<` and `>` are shell redirection unless quoted; applications pin tightly (`==`), libraries stay loose (ranges).

In [ ]:
meanings = {
    "==2.3.1": "exactly this version - hard pin, maximum reproducibility",
    ">=2.2":   "this version or newer - minimum floor",
    "<3":      "strictly below - dodge a breaking major release",
    "~=2.32":  ">=2.32, <3.0 - compatible within the major",
    "!=1.9":   "everything except this buggy release",
}
for spec, meaning in meanings.items():
    print(f"{spec:<10} {meaning}")

# Unquoted, PowerShell reads '>' as output redirection (and rejects '<'
# outright), so the install misbehaves. Quote the whole specifier:
#
# python -m pip install "requests>=2.31,<3"
#
# Rule of thumb: applications pin tightly (==); libraries stay loose
# (ranges) so they can coexist with other packages.